# Training Deteksi Kendaraan Indonesia — Multi-Model

Satu notebook untuk semua varian YOLO. **Ganti satu baris `MODEL` saja**, nama folder,
evaluasi, dan unduhan ikut menyesuaikan sendiri — jadi hasil antar model tidak pernah
tertukar.

Dataset: `puteri-marlisajasmine/deteksi-kendaraan-indonesia-3-ddnbr` versi 3 (Roboflow).

**PENTING sebelum mulai:** menu **Runtime → Change runtime type → T4 GPU → Save**.

Jalankan sel berurutan dari atas.


## 1. Install dependensi

**Jalankan sel ini paling pertama**, sebelum sel apa pun yang meng-`import torch`.


In [ ]:
%pip install -q ultralytics roboflow
import ultralytics; ultralytics.checks()

## 2. Cek GPU

Dijalankan **setelah** install, karena `pip install` mengganti versi torch di disk.
Kalau `import torch` dijalankan sebelum install, torch jadi setengah ter-load dan muncul
`AttributeError: partially initialized module 'torch'` -> obatnya Runtime > Mulai ulang sesi.


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'GPU tidak aktif! Runtime -> Change runtime type -> T4 GPU'
print('GPU siap:', torch.cuda.get_device_name(0))

## 3. Unduh dataset dari Roboflow

API key diminta lewat prompt (tidak tersimpan di notebook, jadi aman kalau notebook dibagikan).
Ambil di Roboflow → Settings → API Keys → **Private API Key**.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

API_KEY = getpass('Roboflow Private API Key: ')

rf = Roboflow(api_key=API_KEY)
version = rf.workspace('puteri-marlisajasmine').project('deteksi-kendaraan-indonesia-3-ddnbr').version(3)
try:
    ds = version.download('yolov11', location='/content/dataset', overwrite=True)
except Exception as e:
    print('format yolov11 gagal:', e, '-> fallback yolov8')
    ds = version.download('yolov8', location='/content/dataset', overwrite=True)

DATA_YAML = f'{ds.location}/data.yaml'
print('\ndata.yaml:', DATA_YAML)
print(open(DATA_YAML).read())

In [ ]:
# Cek jumlah gambar tiap split
import glob, yaml
cfg = yaml.safe_load(open(DATA_YAML))
print('kelas:', cfg['names'])
for split in ['train', 'valid', 'test']:
    n = len(glob.glob(f'{ds.location}/{split}/images/*'))
    print(f'{split:6s}: {n} gambar')

## 4. Training

Ganti `MODEL` di sel bawah, lalu jalankan. Panduan batch untuk T4 (16 GB) @640:

| MODEL | Parameter | BATCH aman | Perkiraan waktu 100 epoch |
|---|---|---|---|
| `yolov8n.pt` | 3.2 jt | 32 | ~25-40 menit |
| `yolov8s.pt` | 11.2 jt | 16 | ~40-60 menit |
| `yolov8m.pt` | 25.9 jt | 8 | ~1.5-2 jam |
| `yolo11n.pt` | 2.6 jt | 32 | ~25-40 menit |
| `yolo11s.pt` | 9.4 jt | 16 | ~40-60 menit |

Kalau kena `CUDA out of memory`, turunkan `BATCH` setengahnya.
Semua parameter lain (epoch, imgsz, augmentasi, `seed=0`) dibiarkan sama antar model
supaya perbandingannya adil — hanya arsitekturnya yang berbeda.


In [ ]:
from ultralytics import YOLO

# ======== CUKUP GANTI BARIS INI ========
MODEL   = 'yolov8s.pt'
BATCH   = 16
# =======================================
EPOCHS  = 100
IMGSZ   = 640

# Nama & path diturunkan otomatis dari MODEL -> tidak akan tertukar antar varian
RUN_NAME = 'kendaraan_' + MODEL.replace('.pt', '')
RUN_DIR  = f'/content/runs/{RUN_NAME}'
BEST     = f'{RUN_DIR}/weights/best.pt'
print('Run   :', RUN_NAME)
print('Folder:', RUN_DIR)

model = YOLO(MODEL)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    workers=2,          # Colab cuma 2 vCPU
    amp=True,           # aman di CUDA, ~2x lebih cepat
    patience=30,        # early stopping
    project='/content/runs',
    name=RUN_NAME,
    exist_ok=True,
    seed=0,
    plots=True,
    # augmentasi khas kamera jalan raya
    degrees=0.0,        # kamera CCTV jarang miring
    fliplr=0.5,
    scale=0.5,          # kendaraan dekat vs jauh
    mosaic=1.0,
    close_mosaic=10,
)


## 5. Evaluasi di test set

In [ ]:
import json, os, time
import pandas as pd

best = YOLO(BEST)
m = best.val(data=DATA_YAML, split='test', device=0)

print(f'\n=== {RUN_NAME} ===')
print(f'mAP50    : {m.box.map50:.4f}')
print(f'mAP50-95 : {m.box.map:.4f}\n')
print(f"{'kelas':<16}{'P':>8}{'R':>8}{'mAP50':>9}{'mAP50-95':>10}")
for i, c in enumerate(m.box.ap_class_index):
    print(f'{best.names[c]:<16}{m.box.p[i]:>8.3f}{m.box.r[i]:>8.3f}'
          f'{m.box.ap50[i]:>9.3f}{m.box.ap[i]:>10.3f}')

# kecepatan inferensi (bahan pembanding untuk skripsi)
spd = m.speed  # ms per gambar
# 'loss' bukan bagian inferensi -> dikeluarkan agar angka fps jujur untuk skripsi
infer_ms = spd['preprocess'] + spd['inference'] + spd['postprocess']
fps = 1000 / infer_ms if infer_ms else 0
print(f'\nkecepatan inferensi: {infer_ms:.1f} ms/gambar (~{fps:.0f} fps di T4)')
print(f'  rincian: preprocess {spd["preprocess"]:.1f} + inference {spd["inference"]:.1f}'
      f' + postprocess {spd["postprocess"]:.1f} ms')

# --- berapa epoch yang benar-benar jalan? (early stopping bisa memotong) ---
res = pd.read_csv(f'{RUN_DIR}/results.csv')
res.columns = [c.strip() for c in res.columns]
ep_berhenti = int(res['epoch'].iloc[-1])
# fitness Ultralytics = 0.1*mAP50 + 0.9*mAP50-95
_fit = 0.1 * res['metrics/mAP50(B)'] + 0.9 * res['metrics/mAP50-95(B)']
ep_terbaik = int(res['epoch'].iloc[_fit.idxmax()])
print(f'\nepoch diminta {EPOCHS}, berhenti di {ep_berhenti}, terbaik di {ep_terbaik}')
if ep_berhenti < EPOCHS:
    print(f'  -> early stopping aktif (patience 30): {ep_berhenti} - {ep_terbaik} = '
          f'{ep_berhenti - ep_terbaik} epoch tanpa perbaikan')
    if ep_berhenti < EPOCHS - 10:
        print('  -> CATATAN: fase close_mosaic (10 epoch terakhir) tidak sempat jalan penuh')

# --- catat ke tabel perbandingan antar model ---
LOG = '/content/perbandingan_model.csv'
row = {
    'model': MODEL, 'epochs_diminta': EPOCHS, 'epoch_berhenti': ep_berhenti,
    'epoch_terbaik': ep_terbaik, 'imgsz': IMGSZ, 'batch': BATCH,
    'mAP50': round(float(m.box.map50), 4), 'mAP50_95': round(float(m.box.map), 4),
    'precision': round(float(m.box.mp), 4), 'recall': round(float(m.box.mr), 4),
    'ms_per_img': round(infer_ms, 2), 'fps_T4': round(fps, 1),
    'params_MB': round(os.path.getsize(BEST) / 1e6, 1),
    'waktu': time.strftime('%Y-%m-%d %H:%M'),
}
for i, c in enumerate(m.box.ap_class_index):
    row[f'mAP50_{best.names[c]}'] = round(float(m.box.ap50[i]), 4)

df = pd.concat([pd.read_csv(LOG), pd.DataFrame([row])]) if os.path.exists(LOG) \
     else pd.DataFrame([row])
df = df.drop_duplicates(subset=['model'], keep='last')
df.to_csv(LOG, index=False)
print('\n=== PERBANDINGAN SEMUA MODEL ===')
display(df)


In [ ]:
# Kurva training + confusion matrix
import os
from IPython.display import Image, display
for f in ['results.png', 'confusion_matrix_normalized.png', 'val_batch0_pred.jpg']:
    p = f'{RUN_DIR}/{f}'
    if os.path.exists(p):
        print(f)
        display(Image(filename=p, width=900))


## 6. Ambil bobot hasil training

Jalankan **salah satu** sel di bawah. Setelah `best.pt` ada di Mac, taruh di
`~/vehicle-counter/runs/<RUN_NAME>/weights/best.pt` — aplikasi Streamlit otomatis
menemukannya di dropdown "Bobot model", jadi kamu bisa membandingkan hasil deteksi
antar varian tanpa mengubah kode.


In [ ]:
# Opsi A: download langsung ke Mac
from google.colab import files
files.download(BEST)


In [ ]:
# Opsi B: simpan ke Google Drive (aman kalau runtime keburu putus)
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
dst = '/content/drive/MyDrive/vehicle-counter'
os.makedirs(dst, exist_ok=True)
shutil.copy(BEST, f'{dst}/best_{RUN_NAME}.pt')
shutil.make_archive(f'{dst}/{RUN_NAME}', 'zip', RUN_DIR)
if os.path.exists('/content/perbandingan_model.csv'):
    shutil.copy('/content/perbandingan_model.csv', f'{dst}/perbandingan_model.csv')
print('tersimpan di Drive:', dst)
print('  best_%s.pt, %s.zip, perbandingan_model.csv' % (RUN_NAME, RUN_NAME))


## 7. (Opsional) Uji cepat di video

Upload satu video jalan raya lewat panel Files di kiri, lalu jalankan sel ini untuk
memastikan deteksi + ByteTrack sudah masuk akal sebelum dipakai menghitung di Mac.

In [ ]:
VIDEO = '/content/jalan.mp4'   # sesuaikan nama file yang kamu upload

import collections
ids_per_kelas = collections.defaultdict(set)

for r in best.track(source=VIDEO, tracker='bytetrack.yaml', persist=True,
                    conf=0.3, imgsz=960, device=0, stream=True,
                    save=True, project='/content/runs', name=f'predict_{RUN_NAME}',
                    exist_ok=True):
    b = r.boxes
    if b is not None and b.id is not None:
        for tid, c in zip(b.id.int().tolist(), b.cls.int().tolist()):
            ids_per_kelas[best.names[c]].add(tid)

print('Perkiraan kendaraan unik (jumlah track ID):')
for k, v in sorted(ids_per_kelas.items()):
    print(f'  {k:<14}{len(v)}')
print(f'\nVideo hasil: /content/runs/predict_{RUN_NAME}/')
print('Catatan: ini hitungan kasar per track ID. Penghitungan resmi berbasis')
print('garis lintasan ada di src/count.py yang jalan di Mac.')
